In [1]:
import numpy as np 
import xarray as xr

#my own classes
import sys
sys.path.append("/emmy-noether/home/mf671/github/blocktrack/lib")
import blocktoolbox as BT

import time
start_time = time.time()

In [2]:
%cd /emmy-noether/home/mf671/DATA/ERA5/geop
!cdo -selmon,6,7,8 ERA5_northem_2.5x2.5_zg500_daily_1940-2023.nc ERA5_northem_2.5x2.5_zg500_daily_jja_1940-2023.nc
!ls

/emmy-noether/home/mf671/DATA/ERA5/geop
cdo    selmonth: Processed 41174784 values from 1 variable over 30681 timesteps [2.78s 123MB]
ERA5_2.5x2.5_zg500_daily_1940-2024.nc
ERA5_northem_2.5x2.5_zg500_daily_1940-2023_deseasonalized.nc
ERA5_northem_2.5x2.5_zg500_daily_1940-2023.nc
ERA5_northem_2.5x2.5_zg500_daily_1960-1963.nc
ERA5_northem_2.5x2.5_zg500_daily_1960.nc
ERA5_northem_2.5x2.5_zg500_daily_jja_1940-2023.nc
ERA5_northem_2.5x2.5_zg500_daily_jja_1960-1970.nc
ERA5_northem_2.5x2.5_zg500_monthly_jja_1940-2023_detrended.nc
ERA5_northem_2.5x2.5_zg500_monthly_jja_1940-2023.nc
grid.txt


In [3]:
print("Starting operation")

'''
data_dir='/emmy-noether/home/mf671/DATA/ERA5/geop/'
fn = data_dir + "ERA5_northem_2.5x2.5_zg500_daily_1940-2023.nc"
work_dic='/emmy-noether/home/mf671/DATA/ERA5/gr_tracking/'
fn_out_unfilt = work_dic + "ERA5_northem_2.5x2.5_zg_GHA_tracked_unfilt_daily_1940-2023_tyrlis_EW.nc"
fn_out = work_dic + "ERA5_northem_2.5x2.5_zg_GHA_tracked_daily_1940-2023_tyrlis_EW.nc"
fn_dic_unfilt = work_dic + "dictionaries/ERA5_northem_2.5x2.5_zg_GHA_tracked_unfilt_daily_1940-2023_DIC_tyrlis_EW.npy"
fn_dic = work_dic + "dictionaries/ERA5_northem_2.5x2.5_zg_GHA_tracked_daily_1940-2023_DIC_tyrlis_EW.npy"
'''
data_dir='/emmy-noether/home/mf671/DATA/ERA5/geop/'
fn = data_dir + "ERA5_northem_2.5x2.5_zg500_daily_jja_1940-2023.nc"
work_dic='/emmy-noether/home/mf671/DATA/ERA5/gr_tracking/'
fn_out_unfilt = work_dic + "ERA5_northem_2.5x2.5_zg_GHA_tracked_unfilt_daily_jja_1940-2023.nc"
fn_out = work_dic + "ERA5_northem_2.5x2.5_zg_GHA_tracked_daily_jja_1940-2023.nc"
fn_dic_unfilt = work_dic + "dictionaries/ERA5_northem_2.5x2.5_zg_GHA_tracked_unfilt_daily_jja_1940-2023_DIC.npy"
fn_dic = work_dic + "dictionaries/ERA5_northem_2.5x2.5_zg_GHA_tracked_daily_jja_1940-2023_DIC.npy"
#'''

Starting operation


In [4]:
ds = xr.load_dataset(fn)
print("Data correctly read")
#ds = BT.GHA(ds,mer_gradient_filter = True, tyrlis_correction=True)
ds = BT.GHA(ds)
print(np.amax(ds['GHA_freq']))
print("GHA function correctly executed")
ds,dic_unfilt = BT.ContourTracking2D(ds,var_name = "GHA")
#saving temporary data
ds.to_netcdf(fn_out_unfilt)
np.save(fn_dic_unfilt,dic_unfilt)

Data correctly read
__Starting a GHA process__
input: zg500 , multiplicative threshold = 1.26


100%|█████████████████████████████████████████████████████████████████| 7638/7638 [00:05<00:00, 1302.84it/s]


<xarray.DataArray 'GHA_freq' ()> Size: 8B
array(18.56884058)
GHA function correctly executed
__Starting a Tracking process__
input: GHA
connected component analysis


100%|██████████████████████████████████████████████████████████████████| 7728/7728 [00:10<00:00, 730.87it/s]


rearranging indexes


100%|█████████████████████████████████████████████████████████████████| 7728/7728 [00:05<00:00, 1409.07it/s]


calculating characteristics
persistence, track, date:


100%|██████████████████████████████████████████████████████████████████| 7728/7728 [00:21<00:00, 364.00it/s]


area and intensity


100%|███████████████████████████████████████████████████████████████████| 7728/7728 [03:04<00:00, 41.95it/s]


number of labels: 12043.0


In [5]:
from importlib import reload
reload(BT)
ds = xr.load_dataset(fn_out_unfilt)
dic_unfilt = np.load(fn_dic_unfilt,allow_pickle=True)
ds2,dic_filt = BT.FilterEvents(ds,dic_unfilt,var_name = "GHA",
                              pers_min = 5,min_avg_area = 2e6)
print("ContourTracking2D function correclty executed")

__Starting a Filtering process__
input: GHA_tracked
applying filters


100%|██████████████████████████████████████████████████████████████| 12043/12043 [00:00<00:00, 51968.85it/s]


rearranging indexes


100%|████████████████████████████████████████████████████████████████| 7728/7728 [00:00<00:00, 16939.14it/s]


number of labels: 965
ContourTracking2D function correclty executed


In [6]:
#saving data
np.save(fn_dic,dic_filt)
ds2.to_netcdf(fn_out)
print("Data created")
  
print("--- %s seconds ---" % (time.time() - start_time))

PermissionError: [Errno 13] Permission denied: '/emmy-noether/home/mf671/DATA/ERA5/gr_tracking/ERA5_northem_2.5x2.5_zg_GHA_tracked_daily_jja_1940-2023.nc'

In [ ]:
label = 20
print(dic_filt[label])
print(sum(dic_filt[label]['WBI'])/len(dic_filt[label]['WBI']))

In [ ]:
WBIS = []
INTENSITIES = []
XS = []
YS = []

for event in dic_filt[1:]:
    for i in range(event['persistence']):
        WBIS.append(sum(event['WBI'])/len(event['WBI']))
        INTENSITIES.append(event['intensity'][i])
        XS.append(event['track'][0][i])
        YS.append(event['track'][1][i])

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# Sample data: latitudes, longitudes, and a float value for color mapping
lats =  XS # Example latitudes
lons = YS#[-118.25, -74.01, -0.12, 151.21, 139.69]  # Example longitudes
values = WBIS#[10, 20, 30, 40, 50]  # Float values for coloring

# Create figure and axis with a projection
fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()})

# Add map features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='lightgray')
ax.set_extent((-100,40,30,90))
vmin = -50 #Plot the scatter points with color mapping
vmax = 50
sc = ax.scatter(lons, lats, c=values, cmap='bwr', vmin=vmin, vmax=vmax ,edgecolors='grey',s=40, transform=ccrs.PlateCarree())

# Add colorbar
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', label='WBI')

# Show the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# Sample data: latitudes, longitudes, and a float value for color mapping
lats =  XS # Example latitudes
lons = YS#[-118.25, -74.01, -0.12, 151.21, 139.69]  # Example longitudes
values = INTENSITIES#[10, 20, 30, 40, 50]  # Float values for coloring

# Create figure and axis with a projection
fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()})

# Add map features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='lightgray')
ax.set_extent((-100,40,30,90))
vmin = np.amin(np.array(values)) #Plot the scatter points with color mapping
vmax = np.amax(np.array(values))
sc = ax.scatter(lons, lats, c=values, cmap='Reds', vmin=vmin, vmax=vmax ,edgecolors='grey',s=40, transform=ccrs.PlateCarree())

# Add colorbar
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', label='Intensity')

# Show the plot
plt.show()


In [ ]:
percentile_90 = np.percentile(INTENSITIES, 95)

In [ ]:
WBIS_max = []
INTENSITIES_max = []
XS_max = []
YS_max = []

for i,intensity in enumerate(INTENSITIES):
    if intensity > percentile_90:
        WBIS_max.append(WBIS[i])
        INTENSITIES_max.append(INTENSITIES[i])
        XS_max.append(XS[i])
        YS_max.append(YS[i])

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# Sample data: latitudes, longitudes, and a float value for color mapping
lats =  XS_max # Example latitudes
lons = YS_max#[-118.25, -74.01, -0.12, 151.21, 139.69]  # Example longitudes
values = INTENSITIES_max#[10, 20, 30, 40, 50]  # Float values for coloring

# Create figure and axis with a projection
fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()})

# Add map features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='lightgray')
ax.set_extent((-100,40,30,90))
#vmin = np.amin(np.array(values)) #Plot the scatter points with color mapping
#vmax = np.amax(np.array(values))
sc = ax.scatter(lons, lats, c=values, cmap='Reds', vmin=vmin, vmax=vmax ,edgecolors='grey',s=40, transform=ccrs.PlateCarree())

# Add colorbar
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', label='Intensity')

# Show the plot
plt.show()
